# Lab08.3 — Traffic-Sign Recognition Using a Binarized Neural Network on FPGA

## Objectives

In this laboratory, you will run a binarized neural network (BNN) accelerator in the programmable logic (PL) of the Zybo board. Python running on the ARM processor (PS) prepares the image, transfers it to the FPGA through AXI DMA, and reads the classification scores.

By the end of the experiment, you will be able to:

- Load an FPGA overlay from Python and identify the AXI DMA interface.
- Prepare a 32 × 32 RGB image for hardware inference.
- Transfer image data to the accelerator and interpret its output scores.
- Compare FPGA predictions with reference labels from the GTSRB dataset.
- Measure end-to-end inference time and identify the contribution of software and data transfers.
- Optionally classify an image captured with a Logitech C270 webcam.

## Hardware and data

- **Board:** Digilent Zybo Legacy (XC7Z010), PYNQ 3.0.1.
- **FPGA design:** `../overlays/compact_gtsrb.bit` with its matching `.hwh` file.
- **Input:** RGB `uint8` image of 32 × 32 pixels (3072 bytes).
- **Output:** 44 signed `int16` scores (88 bytes). Indices 0–42 represent GTSRB classes; index 43 is unused padding and must not be interpreted as a traffic-sign class.
- **Communication:** PS ↔ PL using AXI DMA and contiguous memory buffers.

The first code cell loads the FPGA design. The following sections test DMA communication, classify one labeled traffic sign, evaluate a set of 20 images, and optionally process a webcam frame.

**Required files:** `Lab08/overlays/compact_gtsrb.bit`, `Lab08/overlays/compact_gtsrb.hwh`, `Lab08/tools/gtsrb.py`, and `Lab08/notebooks/gtsrb_batch20_inputs.npz` (20 prepared GTSRB test images with labels). The 20-image test also requires `Lab08/notebooks/BATCH20_CELL.py`. Open this notebook from the `Lab08/notebooks/` directory.


## 1 — Load the implemented FPGA design

In [ ]:
from pathlib import Path
import time
import numpy as np
from pynq import Overlay, allocate

bitfile = Path('../overlays/compact_gtsrb.bit').resolve()
hwhfile = bitfile.with_suffix('.hwh')
if not bitfile.is_file() or not hwhfile.is_file():
    raise FileNotFoundError('Overlay files not found in Lab08/overlays: {} and {}'.format(bitfile, hwhfile))

ol = Overlay(str(bitfile))
if 'axi_dma_0' not in ol.ip_dict:
    raise RuntimeError('DMA axi_dma_0 was not found in the loaded HWH.')
dma = ol.axi_dma_0
print('Loaded FPGA:', bitfile.name)
print('AXI DMA base:', hex(ol.ip_dict['axi_dma_0']['phys_addr']))
print('MM2S 8-bit, S2MM 16-bit, expected frame: 3072 input bytes / 88 output bytes')
print('Initial DMA status: TX=0x{:08X}, RX=0x{:08X}'.format(
    dma.mmio.read(0x04), dma.mmio.read(0x34)))


## 2 — DMA inference function

Receive DMA starts before transmit. Timeout and status checks prevent an infinite wait if a stream transfer fails. The two DMA buffers are reused. After a timeout reload the overlay.

In [ ]:
# The PYNQ allocator returns physically contiguous, DMA-safe memory.
image_buffer = allocate(shape=(3072,), dtype=np.uint8)
score_buffer = allocate(shape=(44,), dtype=np.int16)

def infer_fpga(rgb32, timeout_s=8.0):
    image = np.asarray(rgb32)
    if image.shape != (32, 32, 3) or image.dtype != np.uint8:
        raise ValueError('Expected RGB image: shape=(32,32,3), dtype=uint8')

    t0 = time.perf_counter()
    image_buffer[:] = np.ascontiguousarray(image).reshape(-1)
    image_buffer.flush()
    score_buffer[:] = 0
    score_buffer.flush()

    # S2MM is armed first so the first score cannot be lost.
    dma.recvchannel.transfer(score_buffer)
    dma.sendchannel.transfer(image_buffer)
    deadline = time.monotonic() + timeout_s
    while not (dma.sendchannel.idle and dma.recvchannel.idle):
        if dma.sendchannel.error or dma.recvchannel.error:
            raise RuntimeError('DMA error: TX=0x{:08X}, RX=0x{:08X}'.format(
                dma.mmio.read(0x04), dma.mmio.read(0x34)))
        if time.monotonic() >= deadline:
            raise TimeoutError('DMA timeout: TX=0x{:08X}, RX=0x{:08X}. '
                'Reload overlay before retry; check TLAST/weights/clocks.'.format(
                    dma.mmio.read(0x04), dma.mmio.read(0x34)))

    # The bounded poll above makes both waits return immediately on success.
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    score_buffer.invalidate()
    scores = np.array(score_buffer, dtype=np.int16, copy=True)
    t_ms = (time.perf_counter() - t0) * 1000.0

    winner = int(np.argmax(scores))
    # Class 43 is the unused padding output, NOT a GTSRB traffic-sign class.
    label = winner if winner < 43 else 'INVALID_OUTPUT_43'
    top3 = np.argsort(scores[:43])[-3:][::-1].astype(int).tolist()
    return dict(label=label, raw_argmax=winner, scores=scores,
                top3=[(i, int(scores[i])) for i in top3], elapsed_ms=t_ms)

def show_result(result):
    print('FPGA prediction:', result['label'])
    print('Top 3 valid classes (raw integer scores):', result['top3'])
    print('Padding score (class 43):', int(result['scores'][43]))
    print('End-to-end DMA time: {:.2f} ms'.format(result['elapsed_ms']))
    print('Raw scores:', result['scores'].tolist())
    print('Scores are NOT probabilities.')


## 3 — First hardware transfer

This transfers an artificial test pattern to verify AXI DMA communication. It is not an accuracy test.

In [ ]:
rgb_test = np.zeros((32, 32, 3), dtype=np.uint8)
rgb_test[:, :16, 0] = 255  # red on left
rgb_test[:, 16:, 1] = 255  # green on right
result = infer_fpga(rgb_test)
show_result(result)
assert len(result['scores']) == 44
print('PASS: complete DMA transaction with 44 int16 scores')


## 4 — FPGA inference on one labeled GTSRB test image

The repository includes `gtsrb_batch20_inputs.npz`, containing 20 prepared GTSRB test images and their reference class IDs. Select one image using `SAMPLE_INDEX` (0–19), display it, run FPGA inference and compare the predicted class with the reference label.

The experiment uses local files only. **No internet connection or external dataset download is required.**

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()  # Notebook in Lab08/notebooks/
TOOLS = ROOT / 'tools'
if not (TOOLS / 'gtsrb.py').is_file():
    raise FileNotFoundError('Missing Lab08/tools/gtsrb.py: {}'.format(TOOLS / 'gtsrb.py'))
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
from gtsrb import LABELS

BATCH_FILE = Path('gtsrb_batch20_inputs.npz')
if not BATCH_FILE.is_file():
    raise FileNotFoundError(
        'Missing {}. Copy the complete Lab08/notebooks directory from GitHub.'
        .format(BATCH_FILE.resolve()))

with np.load(BATCH_FILE, allow_pickle=False) as data:
    rgb_images = np.asarray(data['rgb32'])
    true_classes = np.asarray(data['true_classes'])
    filenames = np.asarray(data['filenames'])

if rgb_images.shape != (20, 32, 32, 3) or rgb_images.dtype != np.uint8:
    raise ValueError('Expected 20 RGB uint8 images with shape (20,32,32,3).')
if true_classes.shape != (20,):
    raise ValueError('Expected 20 reference class IDs.')
if not np.all((true_classes >= 0) & (true_classes < len(LABELS))):
    raise ValueError('One or more reference class IDs are invalid.')

SAMPLE_INDEX = 4  # Choose any number from 0 to 19
if not 0 <= SAMPLE_INDEX < len(rgb_images):
    raise IndexError('SAMPLE_INDEX must be between 0 and 19.')

rgb32 = np.array(rgb_images[SAMPLE_INDEX], copy=True)
true_id = int(true_classes[SAMPLE_INDEX])
filename = str(filenames[SAMPLE_INDEX])

plt.figure(figsize=(4, 4))
plt.imshow(rgb32, interpolation='nearest')
plt.title('GTSRB test image (32x32 RGB)')
plt.axis('off')
plt.show()

print('Image:', filename)
print('Ground truth: {} — {}'.format(true_id, LABELS[true_id]))
result = infer_fpga(rgb32)
show_result(result)
pred_id = result['label']
if isinstance(pred_id, int):
    print('Prediction name: {} — {}'.format(pred_id, LABELS[pred_id]))
else:
    print('Prediction name: invalid output (padding class 43)')
print('CORRECT:', pred_id == true_id)


In [ ]:
# Save the selected input, ground-truth label and FPGA output for later analysis.
np.savez_compressed(
    'fpga_compare_single.npz',
    rgb32=rgb32,
    fpga_scores=result['scores'],
    true_class=true_id,
    sample_index=SAMPLE_INDEX
)
print('Saved: fpga_compare_single.npz')


## 5 — Evaluate 20 GTSRB test images

The file `gtsrb_batch20_inputs.npz` contains a prepared sample of 20 labeled traffic-sign images. Run the next cell to display predictions, count correct classifications and measure the mean end-to-end inference time.

The sample contains only 20 images, so its accuracy must not be interpreted as the accuracy of the model on the complete GTSRB test set.

In [ ]:
%run -i BATCH20_CELL.py


## 6 — Display traffic signs for the camera experiment

Use the **Previous** and **Next** buttons to display any of the 20 GTSRB traffic-sign classes directly inside Jupyter. You can show the sign to the Logitech C270 camera on a separate monitor or phone. The SVG illustrations are stored in `Lab08/display/` and are clear at large sizes.

**Important:** These are schematic reference illustrations, not the same photographs used for the 20-image FPGA benchmark. Camera predictions may differ. The sign viewer only displays reference images; it does **not** itself perform FPGA inference. To classify the sign observed by the camera, execute the next section.

In [ ]:
# Interactive sign viewer (runs locally in Jupyter; no internet required).
from IPython.display import display, SVG, clear_output
import ipywidgets as widgets
import json

DISPLAY_ROOT = Path('../display').resolve()
manifest_path = DISPLAY_ROOT / 'manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(
        'Missing Lab08/display/manifest.json. Download the complete Lab08 directory from GitHub.')

with manifest_path.open('r', encoding='utf-8') as f:
    signs = json.load(f)

sign_index = widgets.BoundedIntText(
    value=1, min=1, max=len(signs), description='Sign:',
    layout=widgets.Layout(width='180px'))
previous_button = widgets.Button(description='Previous', icon='arrow-left')
next_button = widgets.Button(description='Next', icon='arrow-right')
sign_output = widgets.Output()

def show_sign(change=None):
    sign = signs[sign_index.value - 1]
    image_path = DISPLAY_ROOT / sign['image']
    if not image_path.is_file():
        raise FileNotFoundError('Missing sign illustration: {}'.format(image_path))
    with sign_output:
        clear_output(wait=True)
        display(widgets.HTML(
            '<b>{}/{} — {} (GTSRB class {})</b>'.format(
                sign_index.value, len(signs), sign['label'], sign['class_id'])))
        display(SVG(filename=str(image_path)))

def change_sign(step):
    sign_index.value = ((sign_index.value - 1 + step) % len(signs)) + 1

previous_button.on_click(lambda _: change_sign(-1))
next_button.on_click(lambda _: change_sign(1))
sign_index.observe(show_sign, names='value')
display(widgets.HBox([previous_button, sign_index, next_button]))
display(sign_output)
show_sign()


## 7 — Optional Logitech C270 camera

Acquire one centered crop from the USB camera, resize on ARM to 32×32 and run the FPGA inference. This is not a trained traffic-sign detector and does not verify ground truth.

In [ ]:
# Logitech C270 — capture and FPGA inference

import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(0, cv2.CAP_V4L2)

try:
    if not cap.isOpened():
        print("Camera unavailable at /dev/video0.")
    else:
        # Configure the Logitech C270 for MJPEG
        cap.set(
            cv2.CAP_PROP_FOURCC,
            cv2.VideoWriter_fourcc(*"MJPG")
        )
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        cap.set(cv2.CAP_PROP_FPS, 15)

        print("Camera opened. Capturing frame...")

        ok, frame = False, None

        for _ in range(3):
            ok, frame = cap.read()
            if ok and frame is not None:
                break

        if not ok or frame is None:
            print("Camera opened but no image was captured.")
        else:
            print("Captured frame:", frame.shape)

            # Center crop
            h, w = frame.shape[:2]
            side = min(h, w)
            x0 = (w - side) // 2
            y0 = (h - side) // 2

            crop_bgr = frame[y0:y0+side, x0:x0+side]
            crop_rgb = cv2.cvtColor(
                crop_bgr, cv2.COLOR_BGR2RGB
            )

            # BNN input: 32x32 RGB
            camera_rgb32 = cv2.resize(
                crop_rgb,
                (32, 32),
                interpolation=cv2.INTER_AREA
            )

            plt.figure(figsize=(6, 4))
            plt.imshow(crop_rgb)
            plt.title("Logitech C270 — captured image")
            plt.axis("off")
            plt.show()

            # FPGA inference
            result = infer_fpga(camera_rgb32)
            show_result(result)

finally:
    cap.release()
    print("Camera released.")


## Interpretation and questions

1. Check that DMA returns 44 signed `int16` scores. Why must score index 43 be excluded from the traffic-sign classes?
2. Compare the prediction for the labeled test image with its ground-truth class. Is the classification correct?
3. How many images are classified correctly in the 20-image sample? What is the measured mean inference time?
4. Which operations contribute to the measured end-to-end time (Python preprocessing, buffer operations, DMA transfers and FPGA execution)? Does this measurement represent the latency of the neural-network hardware alone?
5. Compare the ARM-based experiment in Lab08.2 with the FPGA experiment here. The two notebooks use **different BNN architectures and timing methods**; explain why their execution times cannot be used to calculate a valid same-model FPGA speedup.
6. Why must the matching `.bit` and `.hwh` files be kept together?